# 過擬合與泛化

對應 DLI：02、03。

本課使用小型合成資料或 scikit-learn 內建資料，在 CPU 執行，不需下載原課程資料。依序執行所有儲存格。

[在 Google Colab 執行本課](https://colab.research.google.com/github/Samuel-ncu/AI-Stastic/blob/main/notebooks/06_overfitting_generalization.ipynb) · [閱讀完整背景講義](https://github.com/Samuel-ncu/AI-Stastic/blob/main/docs/units/06-overfitting-generalization.md)

建議順序：先讀概念與手算 → 回答檢核 → 執行程式 → 修改練習。

[背景導讀：從統計到 AI 的模型設計](https://github.com/Samuel-ncu/AI-Stastic/blob/main/docs/statistics-to-ai.md)

## 從圖開始

![模型 A、B、C 的訓練及驗證 MSE 並列，其中 B 的驗證 MSE 最低](https://raw.githubusercontent.com/Samuel-ncu/AI-Stastic/main/docs/figures/06-generalization.svg)

讀出 A、B、C 的驗證誤差，選出這次驗證較好的模型。再看訓練誤差與驗證誤差的差距，說出對 C 的診斷假設。 [閱讀本課講義](https://github.com/Samuel-ncu/AI-Stastic/blob/main/docs/units/06-overfitting-generalization.md)

## 學習目標與先備知識

先理解訓練／驗證切分與 MSE。學完能從訓練及驗證結果提出診斷假設，且不把「模型較大」直接等同「模型較好」。



## 1. 背答案與學規律有什麼不同？

**來源摘要 [S4](https://scikit-learn.org/stable/modules/learning_curve.html)：**偏差描述估計方法的系統性誤差，變異描述對訓練資料改變的敏感程度。訓練與驗證曲線可協助觀察模型複雜度是否合適，但不能取代獨立測試。

**本教材自編比喻：**學生把題目及答案一字不漏背下來，原題能全對；只要換題就失敗。這類比過擬合，但不能用這個比喻取代實際資料評估。



## 2. 資料可以想成規律加雜訊

$$y=f(x)+\epsilon$$

f(x) 是希望學到的規律，$\epsilon$ 表示未由輸入完整說明的變動。本課自編資料使用 sin(3x) 加上常態雜訊；這是已知生成機制，真實專案通常不知道 f。

| 狀態 | 本課曲線的可能表現 | 該查什麼 |
|---|---|---|
| 欠擬合 | 訓練與驗證誤差都偏高 | 特徵是否足夠、模型是否過於受限 |
| 過擬合 | 訓練誤差低、驗證誤差高 | 資料量、雜訊、複雜度與正則化 |
| 較合理的擬合 | 訓練與驗證結果符合需求 | 仍需測試資料與部署檢查 |

這是診斷線索，不是只看一張圖就能定論。訓練不足、資料前處理錯誤及分布差異，也會影響曲線。



## 補充｜從迴歸的轉折理解啟動函數

延續延遲問題。假設服務在佇列不長時每多一筆只增加 2 ms，超過兩筆後額外增加 3 ms。原來一條直線只有一個斜率，畫不出轉折。

從網路的運算看也一樣。將 $h=W_1x+b_1$ 代入第二層 $W_2h+b_2$，可以合併成一個仿射變換 $W_2W_1x+(W_2b_1+b_2)$。光增加這類層數，模型仍畫不出轉折。把非線性函數放在兩層中間，就能改變模型可表示的形狀。

例如加入 ReLU $a(z)=\max(0,z)$：

$$\hat y=5+2x+3\,\mathrm{ReLU}(x-2).$$

當 $x=1$，追加項為 0，預測是 7 ms。當 $x=4$，追加項是 $3(4-2)=6$，預測是 19 ms。這是刻意設定的轉折例。實際網路由資料與損失共同決定權重及偏差，轉折位置通常也可隨參數學習。

### 看輸出範圍、斜率與用途

| 函數 | 形狀與梯度 | 選擇情境 |
|---|---|---|
| ReLU $\max(0,z)$ | 正側斜率為 1，負側為 0 | 一般 CNN 或 MLP 隱藏層的簡單起點。若單元對資料長期落在負側，該路徑梯度可能一直為零 |
| Leaky ReLU | 負側保留小斜率，例如 0.01 | 想保留負側的梯度時，可與 ReLU 做對照 |
| GELU $z\Phi(z)$ | $\Phi$ 為標準常態分布的累積機率。它對輸入做平滑加權 | 現成模型指定 GELU 時依架構使用。自建網路可和 ReLU 在相同條件下比較 |
| tanh | 將輸入壓到 $(-1,1)$，兩端斜率接近 0 | 需要有界、帶正負號的狀態時可考慮，例如部分循環網路 |
| Sigmoid | 將輸入壓到 $(0,1)$，兩端斜率接近 0 | 二元機率輸出或需要介於 0 和 1 的閘門。用在深層隱藏層時要留意飽和 |
| 線性輸出（恆等） | 原樣傳遞輸入 | 實數迴歸常用。若目標有明確的範圍，還要考慮輸出參數化 |

隱藏層的選擇要回答「中間表示需要什麼形狀」。輸出層要回答「預測代表什麼」。同一個網路可以在隱藏層用 ReLU，最後輸出實數、二元 logit 或多類別 logits。比較隱藏層時，固定切分、模型寬度、訓練預算和評估方式，再觀察損失曲線與驗證表現。GELU 用到常態分布的累積函數來定義曲線，輸入資料本身可有其他分布。[來源：GELU 原論文](https://arxiv.org/abs/1606.08415)

**自我檢核。** 在 $x=2$ 前後各多加一筆請求，模型的預測分別多 2 和 5 ms。試把 ReLU 轉折點改為 1，再畫殘差並用留出的資料評估。延遲例子的起點見 [第 05 課迴歸與損失](https://github.com/Samuel-ncu/AI-Stastic/blob/main/docs/units/05-statistical-models-loss.md)，前饋層如何用啟動函數見 [第 10 課 Transformer](https://github.com/Samuel-ncu/AI-Stastic/blob/main/docs/units/10-text-statistics.md)。

DLI 的 ASL CNN 使用 ReLU，原始 Transformer 的前饋層也使用 ReLU。閱讀其他架構時請查其實作。[來源：DLI ASL CNN](https://github.com/NVDLI/fundamentals-of-deep-learning-zh/blob/main/course_content/tutorials/03_asl_cnn.ipynb) · [Transformer 原論文 §3.3](https://arxiv.org/html/1706.03762v7#S3.SS3)

### 補充｜Sigmoid、反向傳播與梯度消失

第 05 課只跨一段預測公式。深層網路中，反向傳播要把多層的局部導數依鏈鎖律相乘。先把每層簡化成一個數字。在實際網路中，特徵向量的梯度傳遞還涉及權重矩陣與各層導數。

$$\frac{\partial L}{\partial h_0}
=\frac{\partial L}{\partial h_m}
\frac{\partial h_m}{\partial h_{m-1}}\cdots\frac{\partial h_1}{\partial h_0}.$$

**梯度消失**是這些乘積讓較早層收到的梯度很小，最佳化器能用來更新那些層的訊號變弱。若每段局部導數都是 0.25，經過六段只剩 $0.25^6=1/4096\approx0.000244$ 倍。這個數值描述簡化例的衰減，實際網路還取決於權重、深度、初始化與資料。

#### 為什麼特別討論 Sigmoid？

Sigmoid 把 $z$ 壓到 0 和 1 之間。令 $p=\sigma(z)$，其導數為 $p(1-p)$。在 $z=0$ 時斜率最大，為 0.25。當 $z$ 很正或很負，輸出接近 1 或 0，曲線在兩端變平，這稱為飽和。例如 $z=-8$ 時，導數約 0.000335。若許多隱藏層串接這種小斜率，早期層就可能難以學習。tanh 同樣有兩端飽和。ReLU 的正側斜率是 1，有助於保留該路徑的梯度，但負側是 0。Leaky ReLU 為負側留小斜率，GELU 則用平滑曲線。這些選擇各有梯度行為，結果仍須用相同訓練條件比較。[Glorot 與 Bengio 的研究](https://proceedings.mlr.press/v9/glorot10a.html) · [GELU 原論文](https://arxiv.org/abs/1606.08415)

#### 二元分類最後一層為什麼仍用 Sigmoid？

因為輸出要代表二元結果的機率，訓練訊號還要看**和哪個損失一起微分**。對硬標籤 $y\in\{0,1\}$，未加權的單筆 BCE 搭配 Sigmoid 對 logit 的導數可化簡為 $p-y$。若真實標籤為 1、$z=-8$，$p\approx0.000335$，梯度接近 −1。這個錯得很有把握的預測，仍會收到明確修正訊號。若同樣用 Sigmoid 機率搭配平方損失，導數為 $2(p-y)p(1-p)$，在這個例子中約 −0.000670。兩者是針對**輸出 logit 的局部梯度**，模型較早的層仍有各自的梯度路徑。實務上把 logits 直接傳給 PyTorch BCEWithLogitsLoss 可得到數值較穩定的計算。[PyTorch BCEWithLogitsLoss](https://docs.pytorch.org/docs/2.15/generated/torch.nn.BCEWithLogitsLoss.html)

**自我檢核。** 早期層梯度很小，增加學習率能否只修好早期層？全域學習率會同時影響不同層。先檢查每層梯度、初始化及模型結構，再決定如何調整。第 10 課會用 $h+F(h)$ 說明殘差連接如何改變相乘的因子。[第 10 課：殘差旁路與梯度](https://github.com/Samuel-ncu/AI-Stastic/blob/main/docs/units/10-text-statistics.md)

## 3. 手算理解「差距」

本教材自編三個模型：

| 模型 | 訓練 MSE | 驗證 MSE |
|---|---:|---:|
| A | 12 | 13 |
| B | 3 | 4 |
| C | 0.2 | 9 |

若任務與切分相同，B 在這次驗證比較較好。C 的訓練／驗證差距為 8.8，大於 B 的 1；但只報差距也不夠，A 的差距同樣只有 1，絕對誤差仍較高。



## 4. 正則化是在做什麼？

第 3 節的模型 C 把訓練 MSE 降到 0.2，換到驗證集卻升到 9。**正則化的目的，是減少模型過度依賴訓練資料中的偶然細節，讓它比較有機會用在新資料上。** 可以先檢查兩個問題。模型是否靠很大的係數或很彎的曲線貼合少數資料點？模型是否太依賴某個中間特徵？L2 懲罰和 Dropout 分別從這兩個地方著手，最後用相同的驗證集確認效果。

### L2 正則化如何改變訓練？

延續第 05 課的迴歸例子，權重決定各個特徵對預測的影響。在輸入尺度固定時，若線性模型的係數很大，特徵稍微變動，預測就可能跟著大幅改變。L2 正則化在原本的訓練損失上，加一項權重平方和：

$$L_{\mathrm{train}}=L_{\mathrm{data}}+\lambda\sum_j w_j^2.$$

$L_{\mathrm{data}}$ 衡量擬合程度，$w_j$ 是被懲罰的第 $j$ 個權重，$\lambda$ 決定願意為較大的權重付多少代價。假設 A、B 的資料損失都是 0.2，並取 $\lambda=0.1$：

| 模型 | 權重 | 權重平方和 | 訓練目標 |
|---|---|---:|---:|
| A | $(1,1)$ | 2 | $0.2+0.1\times2=0.4$ |
| B | $(3,1)$ | 10 | $0.2+0.1\times10=1.2$ |

同樣貼合資料時，這個訓練目標會偏向權重較小的 A。往回算梯度時，平方懲罰會給 $w_j$ 另加一項 $2\lambda w_j$，梯度下降會沿這個項把權重往 0 拉。懲罰很強時，也可能把捕捉真實規律需要的係數壓得太小，造成欠擬合。

Notebook 先改多項式次數，再固定次數和資料，改 `Ridge(alpha=...)` 的懲罰強度。比較時同時看訓練與驗證 MSE。`alpha` 越大，平方懲罰的比重越大，最後以驗證結果選擇適合的設定。[scikit-learn Ridge](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html)

### Dropout 如何讓模型少依賴單一特徵？

**Dropout 在每次訓練的前向計算中，隨機把一部分中間啟動值暫時歸零。** 例如模型辨識手語時，某個中間特徵經常提供關鍵資訊。訓練時偶爾遮掉它，後面的層就會練習在這項資訊缺席時，從其他特徵完成預測。同一組權重持續訓練，每次前向計算重新抽遮罩。[Dropout 原論文](https://jmlr.org/papers/v15/srivastava14a.html)

令 $h$ 為某一個單元的啟動值，$p$ 為丟棄機率。遮罩 $m$ 只有兩種結果，保留時 $m=1$，機率為 $1-p$；丟棄時 $m=0$，機率為 $p$。這就是一次伯努利抽樣（Bernoulli trial）。這裡使用訓練時縮放的做法（inverted dropout），其中 $0\leq p<1$：

$$h_{\mathrm{drop}}=\frac{mh}{1-p}.$$

以 $h=4$、$p=0.5$ 為例，保留時變成 $4/0.5=8$，丟棄時變成 0。長期平均，也就是把兩種結果各乘上其發生機率後相加，得到 $0.5\times8+0.5\times0=4$。**除以保留機率，是為了讓多次抽樣的平均維持在原本的啟動值。** 每一次的輸出仍可能是 0 或 8。

| 階段 | 怎麼處理 $h$ |
|---|---|
| **訓練** | 每次抽遮罩，保留值乘上 $1/(1-p)$。例如 $[2,4,6,8]$ 抽到 $[1,0,1,0]$，輸出為 $[4,0,12,0]$。 |
| **驗證與推論** | 使用評估模式，輸入原樣通過 Dropout。上例輸出為 $[2,4,6,8]$。 |

反向傳播沿著這次前向計算的路徑走。被遮掉的啟動值在該次會乘上 0，因此這條路徑傳回 0；下一次抽到保留時，梯度可以再傳回去。[PyTorch Dropout](https://docs.pytorch.org/docs/2.15/generated/torch.nn.Dropout.html)

**如何判斷有沒有幫助？** 以 $p=0$ 為基準，固定資料切分、模型與訓練預算，再比較不同 $p$ 的驗證表現。若訓練表現下降、驗證表現改善，代表這次遮罩可能減少過擬合。若兩者都變差，可檢查是否遮掉太多有效資訊。Notebook 用重複抽樣核對啟動值的平均與波動。Dropout 能否改善泛化，還要用實際訓練與驗證曲線來判斷。

要找出調整是否有效，作圖前先說明這次改的是懲罰強度或丟棄機率。下一節再區分模型容量、資料筆數與訓練時間這三種橫軸。

## 5. 不同橫軸代表不同問題

- 模型複雜度：改深度或多項式次數，問容量是否合適。
- 訓練資料量：逐步加資料，問更多資料是否有幫助。
- Epoch：看訓練過程，問是否訓練不足或開始過擬合。

DLI 常畫的是 Epoch 曲線；不要把它與「訓練樣本數」曲線當成同一種實驗。Early stopping 可依驗證表現選停止點，之後仍用獨立測試評估。

## 自我檢核

1. 訓練 Accuracy 100%，驗證 60%，應先宣布成功嗎？
2. 訓練及驗證都差時，增加資料一定有用嗎？
3. 用驗證集挑出最佳模型後，還需要測試集嗎？

### 參考解答

1. 不應。先查洩漏、資料差異、過擬合及流程問題。
2. 不一定；特徵不足、模型受限或實作錯誤也可能是原因。
3. 需要；驗證集已參與選擇，不能當作完全未受選擇影響的最後評估。

## 環境

若套件尚未安裝，先執行下一格。安裝需網路，之後的實驗不需下載資料。

In [ ]:
%pip -q install numpy matplotlib scikit-learn

### 比較單一直線與加入 ReLU 特徵的擬合

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

queue_bend = np.arange(5, dtype=float)
latency_bend = np.array([5., 7., 9., 14., 19.])
linear_features = np.column_stack([np.ones(queue_bend.size), queue_bend])
relu_feature = np.maximum(0, queue_bend - 2)
hinge_features = np.column_stack([linear_features, relu_feature])

def fitted_values(features, target):
    coefficients = np.linalg.lstsq(features, target, rcond=None)[0]
    return features @ coefficients

plain_fit = fitted_values(linear_features, latency_bend)
hinge_fit = fitted_values(hinge_features, latency_bend)
print("One slope, residuals:", np.round(latency_bend - plain_fit, 2))
print("ReLU feature, residuals:", np.round(latency_bend - hinge_fit, 2))
np.testing.assert_allclose(hinge_fit, latency_bend)

fig, ax = plt.subplots()
ax.scatter(queue_bend, latency_bend, label="Observed")
ax.plot(queue_bend, plain_fit, label="Single line")
ax.plot(queue_bend, hinge_fit, label="Line + ReLU feature")
ax.set(xlabel="Requests ahead", ylabel="Latency (ms)")
ax.legend()
plt.show()

### 看梯度隨深度變化

以下模擬固定局部斜率的純量例子。帶有旁路的曲線在此例上升，顯示梯度可能放大。真實網路需檢查實際權重、正規化和資料。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

depth = np.arange(1, 13)
# At h=0, centered sigmoid F(h)=sigmoid(h)-0.5 returns zero,
# so every repeated layer is evaluated at the same point.
sigmoid_slope = 0.25
plain_gradient = sigmoid_slope ** depth
skip_gradient = (1 + sigmoid_slope) ** depth
print("At depth 6, plain / residual:", plain_gradient[5], skip_gradient[5])
np.testing.assert_allclose(plain_gradient[5], 1 / 4096)

# A confident wrong positive example at the output.
p = 1 / (1 + np.exp(8.0))
print("Output gradients, BCE / squared error:", p - 1, 2 * (p - 1) * p * (1 - p))

fig, ax = plt.subplots()
ax.semilogy(depth, plain_gradient, "o-", label="Plain: 0.25 per layer")
ax.semilogy(depth, skip_gradient, "s-", label="Residual: 1.25 per layer")
ax.set(xlabel="Number of scalar layers", ylabel="Gradient multiplier")
ax.legend()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
rng = np.random.default_rng(42)
x_train = np.linspace(-1, 1, 20).reshape(-1, 1)
y_train = np.sin(3*x_train[:, 0]) + rng.normal(0, .25, 20)
x_val = np.linspace(-1, 1, 100).reshape(-1, 1)
y_val = np.sin(3*x_val[:, 0]) + rng.normal(0, .25, 100)
for degree in [1, 3, 15]:
    model = make_pipeline(PolynomialFeatures(degree), Ridge(alpha=1e-8))
    model.fit(x_train, y_train)
    print(degree, 'train MSE=', mean_squared_error(y_train, model.predict(x_train)),
          'validation MSE=', mean_squared_error(y_val, model.predict(x_val)))
    plt.plot(x_val, model.predict(x_val), label=f'degree {degree}')
plt.scatter(x_train, y_train); plt.ylim(-3, 3); plt.legend(); plt.show()

### 固定次數，看看 L2 強度如何改變擬合

沿用上一格的資料，固定多項式次數為 15，只改 `alpha`。先猜哪條曲線較貼近訓練點，再用驗證 MSE 評估這個取捨。

In [ ]:
degree = 15
alphas = (1e-8, 0.01, 1.0, 100.0)

fig, ax = plt.subplots()
ax.scatter(x_train[:, 0], y_train, color="black", s=14, label="training samples")
for alpha in alphas:
    model = make_pipeline(PolynomialFeatures(degree), Ridge(alpha=alpha))
    model.fit(x_train, y_train)
    train_mse = mean_squared_error(y_train, model.predict(x_train))
    val_predictions = model.predict(x_val)
    val_mse = mean_squared_error(y_val, val_predictions)
    print(f"alpha={alpha:g}, train MSE={train_mse:.4f}, validation MSE={val_mse:.4f}")
    ax.plot(x_val[:, 0], val_predictions, label=f"alpha={alpha:g}")

ax.set(xlabel="x", ylabel="y", ylim=(-3, 3), title="Fixed degree 15: change only Ridge alpha")
ax.legend()
plt.show()

### 用抽樣核對 Dropout 的平均與波動

先用固定遮罩重現講義的向量例題，再對啟動值 4 重複抽樣。左圖看長期平均，右圖看單次計算的波動。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

hidden = np.array([2.0, 4.0, 6.0, 8.0])
fixed_mask = np.array([1, 0, 1, 0])
masked = hidden * fixed_mask / (1 - 0.5)
np.testing.assert_allclose(masked, [4.0, 0.0, 12.0, 0.0])
print("Training, one draw:", masked)
print("Evaluation:", hidden)

dropout_rng = np.random.default_rng(2026)
rates = [0.0, 0.2, 0.5, 0.8]
activation = 4.0
means, deviations = [], []
for rate in rates:
    keep_probability = 1 - rate
    kept = dropout_rng.random(50_000) < keep_probability
    samples = activation * kept / keep_probability
    means.append(samples.mean())
    deviations.append(samples.std())
    print(f"p={rate:.1f}, mean={means[-1]:.3f}, SD={deviations[-1]:.3f}")

np.testing.assert_allclose(means, activation, atol=0.15)
fig, (ax_mean, ax_sd) = plt.subplots(1, 2, figsize=(9, 3.5))
ax_mean.plot(rates, means, "o-", label="Sample mean")
ax_mean.axhline(activation, linestyle="--", color="gray", label="Evaluation output")
ax_mean.set(xlabel="Drop probability p", ylabel="Mean", ylim=(0, 5.5))
ax_mean.legend()
ax_sd.plot(rates, deviations, "o-", color="tab:orange")
ax_sd.set(xlabel="Drop probability p", ylabel="Standard deviation")
fig.tight_layout()
plt.show()

## 如何解讀

比較曲線與兩組誤差，避免只挑訓練誤差最低者。具體排序受樣本與雜訊影響，應多次重複；此驗證集可選模型，正式測試集仍需另外保留。

## 練習

1. 固定多項式次數與資料，換一個 `alpha`，記錄訓練和驗證 MSE。
2. 把 Dropout 機率加上 0.6。預測輸出平均與標準差會怎麼變，用抽樣和圖表核對。這個實驗驗證的是啟動值的縮放，模型的泛化效果需要另用固定切分比較。

提交修改後的程式、結果圖表及 200 字解讀，列出至少一項限制。若用 AI 協助，說明你如何驗證結果。

## 連結原課程

回到 [DLI Notebook](https://github.com/NVDLI/fundamentals-of-deep-learning-zh/tree/main/course_content/tutorials)，找出本概念對應的資料處理、模型訓練或評估步驟。

## 出處與閱讀建議

- **S4**：[scikit-learn，Validation curves](https://scikit-learn.org/stable/modules/learning_curve.html)。

本課以所列來源的概念作繁體中文摘要，並加入自行編寫的情境、手算、程式及練習；不是原網站的逐段翻譯。標為「來源摘要」者為改寫，標為「自編」者為本教材教學設計；短引文另以引號及「中譯」標明。來源連結可供對照原文，查閱日期：2026-11-17。
